# AmsiNet - Step 11: paired bootstrap of AmsiNet against the flat ViT-B/16

**Runtime:** Colab **A100** (Part A needs the GPU for about 5 minutes; Part B is CPU work in the same session). Needs the five flat checkpoints `ckpt/vit_base_patch16_224_seed0..4.pt` from Step 2, `crops224.tar`, `crops_meta.csv`, and the Step 3a agents.

**Part A** loads each flat ViT-B/16 checkpoint and saves its softmax probabilities on the validation and test words (`agents/flatvit_seed{k}_{split}_probs.npy`). It checks that the accuracy and macro-F1 it recomputes equal the numbers saved in Step 2, so nothing is retrained and nothing about the flat model changes.

**Part B** pairs AmsiNet seed k (ungated cascade at tau*, fusion weights and tau* chosen on validation for that seed) with flat seed k, resamples the 1,283 test **images** (not words) 1,000 times with the same resample for both systems, and reports the difference in accuracy and macro-F1, per seed and averaged over the five seeds, with percentile 95% intervals. It also checks a **non-inferiority margin** on macro-F1 (default 1.0 point), which is the statement the paper needs for "comparable macro-F1 at a fraction of the cost".

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.special import softmax
from scipy import stats
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(SEEDS=[0, 1, 2, 3, 4], VARIANTS=['word', 'word_moe'], EPS=1e-4, KEEP=0.90)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat_df = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); FLAT = float(flat_df[flat_df.agent == 'Flat baseline']['GFLOPs'].iloc[0])
missing = [f'{t}_seed{s}' for s in CFG['SEEDS'] for t in ('block', 'line', 'word', 'word_moe') if not os.path.exists(f'{ROOT}/agents/{t}_seed{s}_test_probs.npy')]
print('missing agent files:', missing or 'none')

In [ ]:
# PART A (GPU): flat ViT-B/16 probabilities, saved once
import torch, timm, subprocess, glob
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import f1_score
LOCAL = '/content/bstd'; DEV = 'cuda'; os.makedirs(LOCAL, exist_ok=True)
CROPS, CROPS_TAR = f'{LOCAL}/crops224', f'{ROOT}/data/crops224.tar'
if not os.path.isdir(CROPS) and os.path.exists(CROPS_TAR): print('restoring crops'); subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)
class SQ(Dataset):
    def __init__(s, p, tf): s.p, s.tf = p, tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert('RGB'))
for seed in CFG['SEEDS']:
    if all(os.path.exists(f'{ROOT}/agents/flatvit_seed{seed}_{sp}_probs.npy') for sp in ('val', 'test')): print('skip (done): flat seed', seed); continue
    ck = f'{ROOT}/ckpt/vit_base_patch16_224_seed{seed}.pt'; assert os.path.exists(ck), 'missing ' + ck + ' | available: ' + str(glob.glob(f'{ROOT}/ckpt/*'))
    m = timm.create_model('vit_base_patch16_224.augreg_in21k', pretrained=False, num_classes=C); m.load_state_dict(torch.load(ck, map_location='cpu')); m = m.to(DEV).eval()
    mc = timm.data.resolve_data_config({}, model=m); tf = T.Compose([T.ToTensor(), T.Normalize(mc['mean'], mc['std'])])
    ref = json.load(open(f'{ROOT}/results/vit_base_patch16_224_seed{seed}.json'))
    for sp in ('val', 'test'):
        d = META[META.split == sp]; out = []
        with torch.no_grad():
            for x in DataLoader(SQ(d.path.tolist(), tf), batch_size=256, num_workers=8):
                with torch.autocast('cuda', dtype=torch.float16): out.append(m(x.to(DEV)).float().softmax(1).cpu())
        P = torch.cat(out).numpy(); y = np.array([SIDX[s] for s in d.script]); pr = P.argmax(1)
        acc, f1 = float((pr == y).mean()), float(f1_score(y, pr, average='macro', labels=list(range(C)), zero_division=0))
        print(f'flat seed {seed} {sp}: acc {acc:.4f} macro-F1 {f1:.4f} | saved in Step 2: acc {ref[sp]["acc"]:.4f} macro-F1 {ref[sp]["macro_f1"]:.4f}')
        assert abs(acc - ref[sp]['acc']) < 0.003 and abs(f1 - ref[sp]['macro_f1']) < 0.005, 'recomputed flat metrics do not match Step 2: stop and tell Claude'
        np.save(f'{ROOT}/agents/flatvit_seed{seed}_{sp}_probs.npy', P.astype(np.float16)); d[['path']].to_csv(f'{ROOT}/agents/flatvit_seed{seed}_{sp}_paths.csv', index=False)
    del m; torch.cuda.empty_cache()
print('flat probabilities ready')

In [ ]:
# 2. Helpers (same definitions as Steps 6 and 8)
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}
def table(sp, wtag, ltag, btag):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(wtag, sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', ltag, lid), block_agent=reg('block', btag, bid),
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=100 * float(np.trace(cm) / cm.sum()), f1=100 * float(f1_from(cm).mean()))
def unc(s): p = softmax(s, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def cascade(T, tau, W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def cost(T, li, bi, GF):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float); nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())
TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]

In [ ]:
RHOS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
def dis_of(g, ng, pred):
    M = np.zeros((ng, C)); np.add.at(M, (g, pred), 1); return (1 - M.max(1) / M.sum(1))[g]          # per word: disagreement of its region
def cascade_g(T, tau, W, rho):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    ok_l = dis_of(T['lg'], T['nl'], s.argmax(1)) <= rho + 1e-9
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], (u > tau) & ok_l); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); ok_b = dis_of(T['bg'], T['nb'], s.argmax(1)) <= rho + 1e-9
    bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], (u > tau) & ok_b); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def fit_W(TV, rho):
    f = lambda W: metr(TV['y'], cascade_g(TV, 0, W, rho)[0].argmax(1))['f1']; W = {k: 0.0 for k in KEYS}
    for _ in range(2):
        for k in KEYS: W[k] = max(GRID_W, key=lambda g: f({**W, k: g}))
    return W, f(W)
def mixed_mask(T):
    first = np.full(T['nl'], -1); mixed = np.zeros(T['nl'], bool)
    for g, yy in zip(T['lg'], T['y']):
        if first[g] < 0: first[g] = yy
        elif first[g] != yy: mixed[g] = True
    return mixed[T['lg']]
def run_g(seed, variant, keep=False):
    GF = dict(word=float(eff[variant]), line=float(eff['line']), block=float(eff['block']))
    wtag, ltag, btag = f'{variant}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
    TV, TT = table('val', wtag, ltag, btag), table('test', wtag, ltag, btag); y = TT['y']; mw = mixed_mask(TT)
    fits = {rho: fit_W(TV, rho) for rho in RHOS}
    best = max(RHOS, key=lambda r: (round(fits[r][1], 6), r))                                  # ties -> larger rho (less gating)
    out = dict(seed=seed, variant=variant, rho_star=best)
    def evaluate(rho, tag):
        W = fits[rho][0]
        cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1))['f1'], cost(TV, r[1], r[2], GF)))(cascade_g(TV, t, W, rho)) for t in TAUS}
        f_full, f_ctx = cv[0][0], cv[1.01][0]; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
        ok = [t for t in TAUS if cv[t][0] >= target]; tau = min(ok, key=lambda t: cv[t][1]) if ok else 0.0
        res = {}
        for name, (s, li, bi) in dict(full=cascade_g(TT, 0, W, rho), cascade=cascade_g(TT, tau, W, rho)).items():
            m = metr(y, s.argmax(1)); out[f'{tag}_{name}_acc'], out[f'{tag}_{name}_f1'], out[f'{tag}_{name}_gflops'] = m['acc'], m['f1'], cost(TT, li, bi, GF)
            out[f'{tag}_{name}_acc_mixed'] = 100 * float((s.argmax(1)[mw] == y[mw]).mean()); out[f'{tag}_{name}_acc_single'] = 100 * float((s.argmax(1)[~mw] == y[~mw]).mean()); res[name] = s.argmax(1)
        out[f'{tag}_tau'] = tau; return res
    base = evaluate(1.0, 'ungated'); gated = evaluate(best, 'gated')
    # oracle upper bound (uses the TRUE mixed-script lines; not a deployable system)
    Wf = fits[1.0][0]; s_all = cascade_g(TT, 0, Wf, 1.0)[0].argmax(1); s_w = TT['word'].argmax(1); orc = np.where(mw, s_w, s_all)
    out['oracle_gate_acc'], out['oracle_gate_f1'] = metr(y, orc)['acc'], metr(y, orc)['f1']
    pk = dict(y=y, word=s_w, full=base['full'], gated=gated['full'], lg=TT['lg'], bg=TT['bg']) if keep else None
    return out, pk

## Part B: AmsiNet predictions per seed, then the paired bootstrap

In [ ]:
BOOT, MARGIN = 1000, 1.0
def amsinet_preds(seed, variant):
    wtag, ltag, btag = f'{variant}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
    GF = dict(word=float(eff[variant]), line=float(eff['line']), block=float(eff['block']))
    TV, TT = table('val', wtag, ltag, btag), table('test', wtag, ltag, btag)
    W, _ = fit_W(TV, 1.0)
    cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1))['f1'], cost(TV, r[1], r[2], GF)))(cascade_g(TV, t, W, 1.0)) for t in TAUS}
    f_full, f_ctx = cv[0][0], cv[1.01][0]; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx); ok = [t for t in TAUS if cv[t][0] >= target]; tau = min(ok, key=lambda t: cv[t][1]) if ok else 0.0
    s, li, bi = cascade_g(TT, tau, W, 1.0)
    return dict(cascade=s.argmax(1), word=TT['word'].argmax(1), full=cascade_g(TT, 0, W, 1.0)[0].argmax(1), y=TT['y'], inv=TT['inv'], n_img=TT['n_img'], gflops=cost(TT, li, bi, GF), tau=tau)
def per_img(p, y, inv, ni):
    M = np.zeros((ni, C * C)); np.add.at(M, (inv, y * C + p), 1); return M
def boot_metrics(cnt, M):
    cms = (cnt @ M).reshape(-1, C, C); acc = np.trace(cms, axis1=1, axis2=2) / cms.sum((1, 2)); tp = np.diagonal(cms, axis1=1, axis2=2); fp = cms.sum(1) - tp; fn = cms.sum(2) - tp
    den = 2 * tp + fp + fn; return 100 * acc, 100 * np.where(den > 0, 2 * tp / np.maximum(den, 1), 0).mean(1)
PRED = {v: {s: amsinet_preds(s, v) for s in CFG['SEEDS']} for v in CFG['VARIANTS']}
FLAT = {}
for s in CFG['SEEDS']:
    P = np.load(f'{ROOT}/agents/flatvit_seed{s}_test_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/flatvit_seed{s}_test_paths.csv').path.tolist()
    assert pa == META[META.split == 'test'].path.tolist(); FLAT[s] = P.argmax(1)
ref = PRED[CFG['VARIANTS'][0]][CFG['SEEDS'][0]]; ni = ref['n_img']; rng = np.random.RandomState(0); cnt = rng.multinomial(ni, np.ones(ni) / ni, size=BOOT).astype(float)
ci = lambda x: (np.percentile(x, 2.5), np.percentile(x, 97.5))
rows = []
for v in CFG['VARIANTS']:
    for sysname in ('cascade', 'full', 'word'):
        dA, dF = [], []
        for s in CFG['SEEDS']:
            r = PRED[v][s]; a1, f1_ = boot_metrics(cnt, per_img(r[sysname], r['y'], r['inv'], ni)); a2, f2_ = boot_metrics(cnt, per_img(FLAT[s], r['y'], r['inv'], ni))
            dA.append(a1 - a2); dF.append(f1_ - f2_)
            if sysname == 'cascade': lo, hi = ci(f1_ - f2_); alo, ahi = ci(a1 - a2); rows.append((v, f'seed {s}', 'cascade - flat', (a1 - a2).mean(), alo, ahi, (f1_ - f2_).mean(), lo, hi))
        dA, dF = np.mean(dA, 0), np.mean(dF, 0); alo, ahi = ci(dA); lo, hi = ci(dF)
        rows.append((v, 'mean of 5 seeds', f'{sysname} - flat', dA.mean(), alo, ahi, dF.mean(), lo, hi))
BT = pd.DataFrame(rows, columns=['word head', 'pairing', 'system minus flat', 'd acc', 'acc CI low', 'acc CI high', 'd macro-F1', 'F1 CI low', 'F1 CI high']); display(BT.round(2)); BT.to_csv(f'{ROOT}/results/paired_bootstrap_vs_flat.csv', index=False)
for v in CFG['VARIANTS']:
    r = BT[(BT['word head'] == v) & (BT.pairing == 'mean of 5 seeds') & (BT['system minus flat'] == 'cascade - flat')].iloc[0]
    print(f"{v}: cascade minus flat, mean of 5 seeds -> accuracy {r['d acc']:+.2f} [{r['acc CI low']:+.2f}, {r['acc CI high']:+.2f}] | macro-F1 {r['d macro-F1']:+.2f} [{r['F1 CI low']:+.2f}, {r['F1 CI high']:+.2f}]")
    print('   macro-F1 non-inferior at margin %.1f point: %s | macro-F1 significantly different from flat: %s | accuracy significantly lower: %s' % (MARGIN, r['F1 CI low'] > -MARGIN, not (r['F1 CI low'] <= 0 <= r['F1 CI high']), r['acc CI high'] < 0))
    print('   GFLOPs per image: cascade %.1f vs flat %.1f' % (np.mean([PRED[v][s]['gflops'] for s in CFG['SEEDS']]), 840.77))